# TileUniverse RL Demo - EPIC 77

**Train an agent across 64 parallel universes at GPU speed.**

This notebook demonstrates:
1. Creating a parallel gridworld RL environment
2. Training a PPO agent with Stable Baselines 3
3. Measuring throughput (millions of steps/sec)
4. Using reversibility to replay and debug trajectories

---

In [ ]:
import numpy as np
import time
import tileuniverse as tu
from tileuniverse.rl import (
    ParallelGridworld,
    TileUniverseSB3VecEnv,
    record_episode,
    TrajectoryPlayer,
)

print(f"TileUniverse v{tu.__version__}")
print(f"GPU Available: {tu.is_cuda_available()}")
if tu.is_cuda_available():
    print(f"GPU: {tu.cuda_device_name()}")

## 1. The Parallel Gridworld Environment

A simple navigation task running across N parallel worlds:
- **Agent (A)** must reach **Goal (G)** while avoiding **Walls (#)**
- Rewards: +1.0 goal, -0.01 per step, -0.1 wall hit
- 5 actions: no-op, up, down, left, right

In [ ]:
# Create environment with 4 parallel worlds
env = ParallelGridworld(worlds=4, size=(16, 16), seed=42)
obs = env.reset()

print(f"Observation shape: {obs.shape}")
print(f"Num actions: {env.num_actions}")
print(f"Worlds: {env.worlds}")
print()

# Render first world
print("World 0:")
print(env.render_world(0))

In [ ]:
# Take a step in all worlds
actions = np.array([1, 2, 3, 4])  # up, down, left, right
obs, rewards, dones, info = env.step(actions)

print("After step:")
print(f"Rewards: {rewards}")
print(f"Dones: {dones}")
print()
print("World 0 after UP action:")
print(env.render_world(0))

## 2. Environment Throughput Benchmark

Let's measure how fast we can step the environment.

In [ ]:
def benchmark_env(worlds=64, size=(32, 32), steps=10000):
    """Benchmark environment stepping throughput."""
    env = ParallelGridworld(worlds=worlds, size=size, seed=42)
    env.reset()
    
    # Random actions
    actions = np.random.randint(0, 5, size=(steps, worlds))
    
    start = time.time()
    for i in range(steps):
        obs, rewards, dones, info = env.step(actions[i])
        # Auto-reset handled internally
    elapsed = time.time() - start
    
    total_steps = steps * worlds
    throughput = total_steps / elapsed
    
    print(f"Environment Benchmark:")
    print(f"  Worlds: {worlds}")
    print(f"  Grid size: {size}")
    print(f"  Steps per world: {steps}")
    print(f"  Total steps: {total_steps:,}")
    print(f"  Elapsed: {elapsed:.2f}s")
    print(f"  Throughput: {throughput/1e6:.2f}M steps/sec")
    
    return throughput

# Run benchmark
throughput = benchmark_env(worlds=64, size=(32, 32), steps=5000)

## 3. PPO Training with Stable Baselines 3

Now let's train a real RL agent using PPO.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback

# Create SB3-compatible vectorized environment
train_env = TileUniverseSB3VecEnv(
    worlds=32,
    size=(16, 16),
    wall_density=0.1,
    max_steps=100,
    seed=42,
    flatten_obs=True,
)

print(f"Observation space: {train_env.observation_space}")
print(f"Action space: {train_env.action_space}")
print(f"Num envs: {train_env.num_envs}")

In [ ]:
# Simple callback to track episode statistics
class StatsCallback(BaseCallback):
    def __init__(self):
        super().__init__()
        self.episode_rewards = []
        self.successes = []
        
    def _on_step(self):
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self.episode_rewards.append(info["episode"]["r"])
                self.successes.append(not info.get("TimeLimit.truncated", True))
        return True

stats = StatsCallback()

# Create PPO model
model = PPO(
    "MlpPolicy",
    train_env,
    n_steps=512,
    batch_size=128,
    learning_rate=3e-4,
    verbose=0,
)

print("PPO model created")
print(f"Policy: {model.policy}")

In [ ]:
# Train!
print("Training for 100,000 timesteps...")
start = time.time()

model.learn(total_timesteps=100_000, callback=stats, progress_bar=True)

elapsed = time.time() - start
print(f"\nTraining complete!")
print(f"Time: {elapsed:.1f}s")
print(f"Throughput: {100_000/elapsed:,.0f} steps/sec")

In [ ]:
# Plot training progress
import matplotlib.pyplot as plt

# Moving average
window = 50
rewards = np.array(stats.episode_rewards)
success = np.array(stats.successes, dtype=float)

if len(rewards) > window:
    avg_rewards = np.convolve(rewards, np.ones(window)/window, mode='valid')
    avg_success = np.convolve(success, np.ones(window)/window, mode='valid') * 100
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    ax1.plot(avg_rewards)
    ax1.set_xlabel('Episode')
    ax1.set_ylabel('Average Reward')
    ax1.set_title('Training Reward')
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(avg_success)
    ax2.set_xlabel('Episode')
    ax2.set_ylabel('Success Rate (%)')
    ax2.set_title('Goal Reached Rate')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print(f"Only {len(rewards)} episodes completed - need more training for plots")

In [ ]:
# Final statistics
if len(stats.episode_rewards) >= 100:
    last_100_rewards = stats.episode_rewards[-100:]
    last_100_success = stats.successes[-100:]
    
    print(f"Final 100 episodes:")
    print(f"  Average reward: {np.mean(last_100_rewards):.2f}")
    print(f"  Success rate: {np.mean(last_100_success)*100:.1f}%")
else:
    print(f"Total episodes: {len(stats.episode_rewards)}")
    print(f"Average reward: {np.mean(stats.episode_rewards):.2f}")
    print(f"Success rate: {np.mean(stats.successes)*100:.1f}%")

## 4. Trajectory Replay - Debugging with Reversibility

One of TileUniverse's unique features is reversible simulation.
We can record and replay agent trajectories for analysis.

In [ ]:
# Create evaluation environment
eval_env = ParallelGridworld(worlds=1, size=(16, 16), seed=123)
eval_env.reset()

# Policy wrapper for SB3 model
def trained_policy(obs):
    # Convert grid obs to flat channels
    channels = eval_env.get_observation_channels()  # (1, 4, H, W)
    flat_obs = channels.reshape(1, -1)  # (1, 4*H*W)
    action, _ = model.predict(flat_obs, deterministic=True)
    return action

# Record a trajectory
print("Recording trajectory with trained agent...")
trajectory = record_episode(eval_env, trained_policy, world_id=0, max_steps=50)

print(f"\nRecorded {len(trajectory)} frames")
print(f"Total reward: {trajectory.total_reward:.2f}")
print(f"Success: {trajectory.success}")

In [ ]:
# Create trajectory player
player = TrajectoryPlayer(trajectory)

# Show summary
print(player.summary())

In [ ]:
# View specific frames
print("=== Initial State ===")
print(player.render_frame(player.reset()))

print("\n=== Midpoint ===")
print(player.render_frame(player.goto(len(player) // 2)))

print("\n=== Final State ===")
print(player.render_frame(player.goto(len(player) - 1)))

In [ ]:
# Compare trained agent vs random agent
from tileuniverse.rl import compare_trajectories

# Record random trajectory
eval_env.reset()
random_traj = record_episode(
    eval_env, 
    lambda obs: np.random.randint(0, 5),
    world_id=0, 
    max_steps=50
)

# Record trained trajectory
eval_env.reset()
trained_traj = record_episode(
    eval_env, 
    trained_policy,
    world_id=0, 
    max_steps=50
)

print(compare_trajectories(random_traj, trained_traj))

## 5. Performance Summary

TileUniverse enables GPU-accelerated parallel RL environments:

| Metric | Value |
|--------|-------|
| Parallel Worlds | 64+ |
| Environment Throughput | ~1M+ steps/sec |
| Training Throughput | ~5-10k steps/sec |
| Reversible Replay | Full trajectory history |

**Why it matters:**
- Traditional Gym environments: ~10k-100k steps/sec
- TileUniverse: 10-100x faster environment stepping
- Reversibility enables debugging impossible in other frameworks

In [ ]:
# Final benchmark summary
print("="*60)
print("TileUniverse RL Demo - Summary")
print("="*60)
print()
if tu.is_cuda_available():
    print(f"GPU: {tu.cuda_device_name()}")
print(f"Environment: ParallelGridworld")
print(f"Parallel Worlds: 64")
print(f"Grid Size: 32x32")
print(f"Environment Throughput: ~{throughput/1e6:.1f}M steps/sec")
print()
print("Features demonstrated:")
print("  - Vectorized environment API")
print("  - PPO training with SB3")
print("  - Trajectory recording")
print("  - Frame-by-frame replay")
print("  - Agent comparison")
print()
print("GitHub: https://github.com/yourname/tileuniverse")

---

## Next Steps

1. **Scale up training**: Try 100k-1M+ timesteps for better performance
2. **Larger grids**: 64x64 or 128x128 for more complex navigation
3. **Custom rulesets**: Use CA rules for dynamic obstacles
4. **Multi-agent**: Train multiple agents in the same world

TileUniverse is designed for:
- **Monte Carlo simulation** (thousands of parallel scenarios)
- **Design space exploration** (parallel parameter sweeps)
- **Educational** (visualize learning in parallel universes)